# Aromas piloto 2026 — captura A+B completa asumida

## tl;dr

Este notebook prueba explícitamente el contrato de diseño `eta_AB = 1`. La eficiencia física no se estima: toda masa que abandona el estado de línea se asigna al condensado combinado A+B. La validación leave-one-run-out determina si producción, partición de Mouret/Morakul y captura completa son conjuntamente compatibles con los datos.

## Contexto y métodos

### Supuestos clave

- A y B se observan químicamente como un único `MIX`; sólo sus volúmenes están separados.
- La captura total A+B se fija en 1 por supuesto de diseño, no por medición.
- La volatilización usa el equilibrio `K(E,T)·QCO2` de Morakul/Mouret.
- El condensado participa en el ajuste como medición de toda la pérdida gaseosa bajo este contrato.
- La validación deja fuera una fermentación completa por fold.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if not (ROOT / 'fermentation_model').exists():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT / 'fermentation_model'))
from pilot_2026 import run_aroma_assumed_complete_capture_validation_2026 as analysis

result = analysis.load_results()
print('Resultados:', analysis.RESULTS_DIR.relative_to(ROOT))
print('Veredicto:', result['gate']['verdict'])
print('Modelo diagnóstico:', result['gate']['diagnostic_best_model'])

## Resultados

In [ ]:
primary_metrics = result['metrics'].query("analysis_policy == 'primary'")
display(primary_metrics.round(4))
display(Image(filename=analysis.FIGURE_DIR / '02_loro_nrmse_comparison.png'))

In [ ]:
display(result['capture_audit'])
display(result['previous_comparison'].round(4))
display(result['parameters'].round(5))

In [ ]:
display(result['mass'].round(4))
assert result['capture_audit']['fixed_total_capture_fraction'].eq(1.0).all()
assert result['capture_audit']['minimum_simulated_capture_fraction'].eq(1.0).all()
assert result['capture_audit']['maximum_simulated_capture_fraction'].eq(1.0).all()
assert result['capture_audit']['maximum_absolute_unrecovered_ug'].max() < 1e-8
assert result['capture_audit']['maximum_absolute_mass_closure_error_ug'].max() < 1e-6

In [ ]:
for name in [
    '03_rco2_temperature_pulse_drivers.png',
    '04_liquid_ethyl_octanoate.png',
    '04_liquid_isoamyl_acetate.png',
    '05_condensate_ethyl_octanoate.png',
    '05_condensate_isoamyl_acetate.png',
    '06_parameter_stability.png',
]:
    display(Image(filename=analysis.FIGURE_DIR / name))

## Conclusiones

El veredicto se interpreta sobre el contrato conjunto, no sobre la eficiencia del equipo. Si el condensado no cierra, los datos actuales no permiten asignar la discrepancia exclusivamente a producción, partición o captura completa. El resultado correcto es aceptar o rechazar la compatibilidad del supuesto, manteniéndolo visible.